# Convolutional Neural Network (PyTorch GPU Implementation)

### Importing the Libraries & Configuring the GPU

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Part 1 - Data Preprocessing

### Defining Image Augmentation & Normalization

In [17]:
train_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, shear=0.2, scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

test_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

### Loading the Training and Test Sets

In [18]:
train_data = datasets.ImageFolder('Dataset/training_set', transform=train_transforms)
test_data = datasets.ImageFolder('Dataset/test_set', transform=test_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

## Part 2 - Building the CNN

### Defining the 3-Layer Convolutional Architecture & Optimizer

In [19]:
class ImprovedCatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            # Block 1: 128x128 -> 64x64
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Block 2: 64x64 -> 32x32
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Block 3: 32x32 -> 16x16
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),  # Randomly drops 50% of neurons to stop overfitting
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x))

model = ImprovedCatDogCNN().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Part 3 - Training the CNN

### Running the Training & Validation Loop (25 Epochs)


In [21]:
for epoch in range(25):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.float().unsqueeze(1).to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        predicted = (outputs > 0.5).float()
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        
        if (batch_idx + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/25] - Batch [{batch_idx+1}/{len(train_loader)}] - Loss: {loss.item():.4f}")

Epoch [1/25] - Batch [50/250] - Loss: 0.6489
Epoch [1/25] - Batch [100/250] - Loss: 0.6662
Epoch [1/25] - Batch [150/250] - Loss: 0.6037
Epoch [1/25] - Batch [200/250] - Loss: 0.6996
Epoch [1/25] - Batch [250/250] - Loss: 0.6604
Epoch [2/25] - Batch [50/250] - Loss: 0.7521
Epoch [2/25] - Batch [100/250] - Loss: 0.6703
Epoch [2/25] - Batch [150/250] - Loss: 0.7350
Epoch [2/25] - Batch [200/250] - Loss: 0.7271
Epoch [2/25] - Batch [250/250] - Loss: 0.6270
Epoch [3/25] - Batch [50/250] - Loss: 0.6237
Epoch [3/25] - Batch [100/250] - Loss: 0.7692
Epoch [3/25] - Batch [150/250] - Loss: 0.5322
Epoch [3/25] - Batch [200/250] - Loss: 0.5637
Epoch [3/25] - Batch [250/250] - Loss: 0.6602
Epoch [4/25] - Batch [50/250] - Loss: 0.5121
Epoch [4/25] - Batch [100/250] - Loss: 0.6529
Epoch [4/25] - Batch [150/250] - Loss: 0.6741
Epoch [4/25] - Batch [200/250] - Loss: 0.6361
Epoch [4/25] - Batch [250/250] - Loss: 0.7647
Epoch [5/25] - Batch [50/250] - Loss: 0.6922
Epoch [5/25] - Batch [100/250] - Loss: 

In [26]:
model.eval()
val_loss, val_correct, val_total = 0.0, 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.float().unsqueeze(1).to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
            
        val_loss += loss.item()
        predicted = (outputs > 0.5).float()
        val_total += labels.size(0)
        val_correct += (predicted == labels).sum().item()
            
avg_train_loss = train_loss / len(train_loader)
avg_val_loss = val_loss / len(test_loader)
train_acc = train_correct / train_total
val_acc = val_correct / val_total

print(f"---> Epoch {epoch+1}/25 Complete | "
        f"Loss: {train_loss/len(train_loader):.4f} - Acc: {train_correct/train_total:.4f} | "
        f"Val Loss: {val_loss/len(test_loader):.4f} - Val Acc: {val_correct/val_total:.4f}\n")

---> Epoch 25/25 Complete | Loss: 0.4227 - Acc: 0.8110 | Val Loss: 0.3903 - Val Acc: 0.8310



In [27]:
    saved_msg = ""
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'cat_dog_pytorch_v2_best.pth')
        saved_msg = " [Saved Best Model]"
        
    print(f"Epoch {epoch+1:02d}/25 | "
          f"Loss: {avg_train_loss:.4f} - Acc: {train_acc:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.4f}{saved_msg}")

Epoch 25/25 | Loss: 0.4227 - Acc: 0.8110 | Val Loss: 0.3903 - Val Acc: 0.8310 [Saved Best Model]


## Part 4 - Making Single Predictions

In [28]:
model.eval()
img_path = 'single_prediction/cat_or_dog_1.jpg'
image = Image.open(img_path).convert('RGB')

input_tensor = test_transforms(image).unsqueeze(0).to(device)

with torch.no_grad():
    probability = model(input_tensor).item()

class_names = train_data.classes
prediction = class_names[1] if probability > 0.5 else class_names[0]

print(f"Class Mapping: 0 -> {class_names[0]}, 1 -> {class_names[1]}")
print(f"Prediction: {prediction} (Probability score: {probability:.4f})")

Class Mapping: 0 -> cats, 1 -> dogs
Prediction: cats (Probability score: 0.2592)


In [29]:
model.eval()
img_path = 'single_prediction/cat_or_dog_2.jpg'
image = Image.open(img_path).convert('RGB')

input_tensor = test_transforms(image).unsqueeze(0).to(device)

with torch.no_grad():
    probability = model(input_tensor).item()

class_names = train_data.classes
prediction = class_names[1] if probability > 0.5 else class_names[0]

print(f"Class Mapping: 0 -> {class_names[0]}, 1 -> {class_names[1]}")
print(f"Prediction: {prediction} (Probability score: {probability:.4f})")

Class Mapping: 0 -> cats, 1 -> dogs
Prediction: dogs (Probability score: 0.9797)


In [30]:
model.eval()
img_path = 'single_prediction/cat_or_dog_3.jpg'
image = Image.open(img_path).convert('RGB')

input_tensor = test_transforms(image).unsqueeze(0).to(device)

with torch.no_grad():
    probability = model(input_tensor).item()

class_names = train_data.classes
prediction = class_names[1] if probability > 0.5 else class_names[0]

print(f"Class Mapping: 0 -> {class_names[0]}, 1 -> {class_names[1]}")
print(f"Prediction: {prediction} (Probability score: {probability:.4f})")

Class Mapping: 0 -> cats, 1 -> dogs
Prediction: dogs (Probability score: 0.9344)
